# Agent 第 14 课：Action Groups —— 让 Bedrock Agent 真的动手

上一课的 agent 只会聊天。这一课给它装"手"：**Action Group**。

一个 Action Group = **一组工具 + 它们的实现（Lambda 或 Return Control）**。Bedrock Agent 的 orchestration 循环每一轮会决定"要不要调某个 action"，调了就等 Lambda 返回结果，继续循环直到给出答复。

学完这节你能：
1. 用 **function schema**（比 OpenAPI 简单得多）声明工具
2. 写一个 Lambda 处理 Bedrock Agent 的调用格式
3. 知道 **Return Control** 是什么、什么时候用它代替 Lambda

## 1. 两种 schema：OpenAPI vs Function Schema

Bedrock Agents 支持两种工具描述：

**OpenAPI 3.0**（老路子）
- 一个 YAML/JSON 文件描述 REST 路径
- 适合"已经有现成 OpenAPI 文档"的内部 API
- 冗长，但能复用

**Function Schema**（推荐，2024 年后加入）
- 直接列函数名 + 参数 + description
- 跟 OpenAI / Converse 的工具声明几乎一样
- **这是 2026 年写新 agent 的默认选择**

下面都用 function schema。

In [ ]:
import boto3, os, json, uuid, time, io, zipfile
os.environ.setdefault("AWS_REGION", "us-west-2")

agent_client = boto3.client("bedrock-agent")
runtime      = boto3.client("bedrock-agent-runtime")
lam          = boto3.client("lambda")
iam          = boto3.client("iam")
sts          = boto3.client("sts")
ACCOUNT = sts.get_caller_identity()["Account"]
REGION  = boto3.session.Session().region_name or "us-west-2"
print("account =", ACCOUNT, "region =", REGION)

## 2. 写一个工具 Lambda

Bedrock 调你的 Lambda 时，event 长这样：
```python
{
    "actionGroup": "WeatherTools",
    "function": "get_weather",              # 被调的函数名
    "parameters": [                          # 参数列表（不是 dict！）
        {"name": "city", "type": "string", "value": "Seattle"}
    ],
    "messageVersion": "1.0",
    "agent": {...}, "sessionId": "...", "sessionAttributes": {...}
}
```

Lambda 必须返回**特定格式**，不然 agent 不认。最常用的是 functionResponse 格式。

In [ ]:
# Lambda 源码（写进 zip，上传）
LAMBDA_SRC = '''
import json

def get_weather(city):
    data = {"Seattle":(11,"rainy"), "Beijing":(18,"smoggy"),
            "Tokyo":(15,"clear"), "SanFrancisco":(14,"foggy")}
    t, cond = data.get(city.replace(" ",""), (20,"unknown"))
    return {"city": city, "temp_c": t, "condition": cond}

def get_time(city):
    tz = {"Seattle":"07:30","Beijing":"22:30","Tokyo":"23:30"}.get(city.replace(" ",""),"12:00")
    return {"city": city, "local_time": tz}

FUNCS = {"get_weather": get_weather, "get_time": get_time}

def lambda_handler(event, context):
    fn_name = event["function"]
    params = {p["name"]: p["value"] for p in event.get("parameters", [])}
    try:
        result = FUNCS[fn_name](**params)
        body = {"TEXT": {"body": json.dumps(result)}}
    except Exception as e:
        body = {"TEXT": {"body": f"Error: {e}"}}
    return {
        "messageVersion": "1.0",
        "response": {
            "actionGroup": event["actionGroup"],
            "function": fn_name,
            "functionResponse": {"responseBody": body},
        },
    }
'''
# 打包
buf = io.BytesIO()
with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as z:
    z.writestr("lambda_function.py", LAMBDA_SRC)
buf.seek(0)
LAMBDA_ZIP = buf.read()
print("zip bytes:", len(LAMBDA_ZIP))

In [ ]:
# 创建（或更新）Lambda —— 假定已有一个 lambda execution role
LAMBDA_ROLE_ARN = f"arn:aws:iam::{ACCOUNT}:role/service-role/demo-lambda-basic"
FN_NAME = "bedrock-agent-tools-demo"

try:
    lam.create_function(
        FunctionName=FN_NAME,
        Runtime="python3.11",
        Role=LAMBDA_ROLE_ARN,
        Handler="lambda_function.lambda_handler",
        Code={"ZipFile": LAMBDA_ZIP},
        Timeout=15,
    )
    print("created lambda")
except lam.exceptions.ResourceConflictException:
    lam.update_function_code(FunctionName=FN_NAME, ZipFile=LAMBDA_ZIP)
    print("updated lambda code")

LAMBDA_ARN = lam.get_function(FunctionName=FN_NAME)["Configuration"]["FunctionArn"]
print("lambdaArn =", LAMBDA_ARN)

# 给 Bedrock 服务加 invoke 权限（幂等处理）
try:
    lam.add_permission(
        FunctionName=FN_NAME,
        StatementId="bedrock-agent-invoke",
        Action="lambda:InvokeFunction",
        Principal="bedrock.amazonaws.com",
        SourceAccount=ACCOUNT,
    )
    print("added lambda permission")
except lam.exceptions.ResourceConflictException:
    print("lambda permission already set")

## 3. 创建 Agent + Action Group

In [ ]:
AGENT_ROLE_ARN = f"arn:aws:iam::{ACCOUNT}:role/AmazonBedrockExecutionRoleForAgents_Demo"
FOUNDATION_MODEL = "anthropic.claude-sonnet-4-6-20260101-v1:0"

resp = agent_client.create_agent(
    agentName=f"weather-agent-{uuid.uuid4().hex[:6]}",
    agentResourceRoleArn=AGENT_ROLE_ARN,
    foundationModel=FOUNDATION_MODEL,
    instruction="You answer weather / time questions. ALWAYS use the tools; never guess.",
    idleSessionTTLInSeconds=600,
)
AGENT_ID = resp["agent"]["agentId"]
print("agentId =", AGENT_ID)

# 等 CREATING 结束再建 Action Group
while agent_client.get_agent(agentId=AGENT_ID)["agent"]["agentStatus"] == "CREATING":
    time.sleep(2)

In [ ]:
# Function Schema（本课重点）
function_schema = {
    "functions": [
        {
            "name": "get_weather",
            "description": "Get current weather (temp in Celsius and condition) for a city.",
            "parameters": {
                "city": {"type": "string", "description": "City name", "required": True},
            },
        },
        {
            "name": "get_time",
            "description": "Get current local time for a city.",
            "parameters": {
                "city": {"type": "string", "description": "City name", "required": True},
            },
        },
    ]
}

ag_resp = agent_client.create_agent_action_group(
    agentId=AGENT_ID,
    agentVersion="DRAFT",
    actionGroupName="WeatherTools",
    actionGroupExecutor={"lambda": LAMBDA_ARN},
    functionSchema=function_schema,
    description="Weather / time lookup tools",
)
print("actionGroupId =", ag_resp["agentActionGroup"]["actionGroupId"])

In [ ]:
# Prepare + Alias（跟上一课一样）
agent_client.prepare_agent(agentId=AGENT_ID)
while agent_client.get_agent(agentId=AGENT_ID)["agent"]["agentStatus"] != "PREPARED":
    time.sleep(3)

AGENT_ALIAS_ID = agent_client.create_agent_alias(
    agentId=AGENT_ID, agentAliasName="dev",
)["agentAlias"]["agentAliasId"]
print("ready:", AGENT_ID, AGENT_ALIAS_ID)

In [ ]:
def ask(text):
    stream = runtime.invoke_agent(
        agentId=AGENT_ID, agentAliasId=AGENT_ALIAS_ID,
        sessionId=str(uuid.uuid4()), inputText=text, enableTrace=True,
    )
    out, tool_calls = [], []
    for ev in stream["completion"]:
        if "chunk" in ev:
            out.append(ev["chunk"]["bytes"].decode())
        elif "trace" in ev:
            t = ev["trace"]["trace"]
            orch = t.get("orchestrationTrace", {})
            if "invocationInput" in orch:
                ii = orch["invocationInput"]
                if "actionGroupInvocationInput" in ii:
                    a = ii["actionGroupInvocationInput"]
                    tool_calls.append((a.get("function"), a.get("parameters")))
    return "".join(out), tool_calls

answer, calls = ask("北京和东京现在的天气和时间分别是什么？")
print("tools called:", calls)
print("answer:", answer)

## 4. Return Control —— 不经 Lambda 的 action

有时候你不想（或不能）把工具实现放在 Lambda 里：
- 工具需要**用户当前的 UI 状态**（前端才知道的东西）
- 工具需要**用户点一下确认**（回到第 10 课 human-in-the-loop）
- 工具是**另一个后端服务**，你不想额外建 Lambda proxy

这时候把 Action Group 的 executor 设为 **`returnControl`** 而不是 lambda。Agent 遇到工具调用时**不执行**，而是把 `returnControl` payload 从 `invoke_agent` 流式返回给你——**你在调用方自己执行工具，然后把结果发回去**继续对话。

```python
actionGroupExecutor={"customControl": "RETURN_CONTROL"}
```

然后在 `invoke_agent` 返回的事件流里处理 `returnControl` 事件，再次调 `invoke_agent`，传 `sessionState.invocationId` + `returnControlInvocationResults`。

**使用 return control 的场景**：前端驱动的 agent、human-in-the-loop、跨服务桥接。
**不用 return control**：纯后端工具（写 Lambda 简单可靠）。

## 5. Schema 设计的几个教训

做了几个生产 agent 后的几条规律：

1. **函数粒度不要太粗**。别搞一个 `execute_action(action_type, payload)` 大杂烩；拆成 `create_ticket` / `update_ticket` / `assign_ticket` 模型调得准得多。
2. **参数 description 写清楚**。"date in ISO 8601 YYYY-MM-DD format" 比 "date" 命中率高一大截。
3. **返回 JSON 别返回 HTML**。文本要尽量结构化，模型才能 chain。
4. **错误也要返回结构化**。`{"error": "city not found", "suggested": ["Seattle", ...]}` 让 agent 能自我纠错。
5. **不要在 Lambda 里做重活**。Lambda timeout 默认 15s（Bedrock 要求 ≤ 30s）；长任务用 step function / 异步 + 返回 job_id。

## 6. 清理

In [ ]:
try:
    agent_client.delete_agent_alias(agentId=AGENT_ID, agentAliasId=AGENT_ALIAS_ID)
except Exception as e: print("skip alias:", e)
try:
    agent_client.delete_agent(agentId=AGENT_ID, skipResourceInUseCheck=True)
except Exception as e: print("skip agent:", e)
try:
    lam.delete_function(FunctionName=FN_NAME)
except Exception as e: print("skip lambda:", e)
print("cleanup done")

## 7. 工程直觉

1. **优先 function schema，不要 OpenAPI**——除非你要复用现成文档。
2. **Lambda 返回必须严格匹配 functionResponse 格式**，字段错一个 agent 就 retry 到死。
3. **Return Control 是 human-in-the-loop 的 AWS 原生实现**，比自己在 Lambda 里卡 SQS 优雅。
4. **工具函数里务必幂等 / 可重试**——Bedrock Agent 会 retry。
5. **下一课**：Knowledge Bases，给 agent 挂文档。你会发现 RAG 在托管服务里配置远简单于自己搓。